#Differential Isoform Usage in Senescence

Identify isoforms with significantly shifted usage (isoform fractions) between senescent (CDKN2A+) and non-senescent (CDKN2A-) cells, and between p16+ and p14ARF+ cells.

Mirrors notebook 04_02–04_11_12 (differential isoform usage across cell types) but focuses specifically on senescence status.

In [1]:
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)

import os
_r = os.path.abspath(".")
while _r != os.path.dirname(_r) and not os.path.exists(os.path.join(_r, ".notebooks_root")):
    _r = os.path.dirname(_r)
os.chdir(_r if os.path.exists(os.path.join(_r, ".notebooks_root")) else "/oak/stanford/groups/quake/mmantri/group.quake/tabula_longread/notebooks")  # cd to notebooks/ root (marked .notebooks_root); relocation-proof

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse
from scipy import stats
import importlib
import warnings, os, sys, gc
warnings.filterwarnings('ignore')
gc.enable()

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
# Reload differential_isoform_fraction so notebook picks up edits (e.g. the
# process-parallel gene-level DM) without restarting the kernel.
import differential_isoform_fraction as dif
importlib.reload(dif)
from differential_isoform_fraction import (
    rank_isoform_fractions,
    get_rank_isoform_fractions_df,
    plot_rank_isoform_fractions,
    plot_isoform_fraction_comparison,
    rank_gene_isoform_usage_dm,
    rank_gene_isoform_usage_dm_by_celltype,
    plot_gene_isoform_dm_volcano,
)
# Reload pyVolcano so notebook picks up edits without restarting the kernel
import pyVolcano
importlib.reload(pyVolcano)
from pyVolcano import plot_volcano, plot_volcano_with_colors
from add_classification import add_classification

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

BASEDIR = "./../pacbio"
H5AD_DIR = f"{BASEDIR}/h5ads"

# reload figure_paths: a live kernel caches FIG_MAP
import importlib
import figure_paths
importlib.reload(figure_paths)
from figure_paths import figpath  # routes figures/ -> figures/figureN/ via FIG_MAP


In [2]:
# preferred plot formatting
import matplotlib
import matplotlib.legend as _mlegend
import matplotlib.pyplot as plt

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial'] + matplotlib.rcParams['font.sans-serif']
# Mathtext (anything in $...$) otherwise falls back to DejaVu Sans, so Greek letters,
# superscripts and subscripts render in a different family from the surrounding Arial.
# "custom" + the four Arial families keeps the whole figure in one typeface.
matplotlib.rcParams['mathtext.fontset'] = 'custom'
matplotlib.rcParams['mathtext.rm'] = 'Arial'
matplotlib.rcParams['mathtext.it'] = 'Arial:italic'
matplotlib.rcParams['mathtext.bf'] = 'Arial:bold'

TICK_LEN = 2.0
TICK_LABEL_PAD = 1.5
AXIS_LABEL_PAD = 1.5
PANEL_TITLE_PAD = 3.0

plt.rcParams.update({
    "font.size": 6,
    "axes.titlesize": 6,
    "axes.labelsize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "legend.labelspacing": 0.1,
    "legend.borderpad": 0.3,
    "legend.handletextpad": 0.2,
    "legend.title_fontsize": 6,
    "figure.titlesize": 6,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
    "xtick.major.size": TICK_LEN,
    "ytick.major.size": TICK_LEN,
    "xtick.major.pad": TICK_LABEL_PAD,
    "ytick.major.pad": TICK_LABEL_PAD,
    "axes.labelpad": AXIS_LABEL_PAD,
    "axes.titlepad": PANEL_TITLE_PAD,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Legend-title weight has no rcParam -> make every legend title bold by default.
if not getattr(_mlegend.Legend, "_bold_title_patched", False):
    _orig_set_title = _mlegend.Legend.set_title
    def _bold_set_title(self, title, prop=None):
        res = _orig_set_title(self, title, prop=prop)
        self.get_title().set_fontweight("bold")
        return res
    _mlegend.Legend.set_title = _bold_set_title
    _mlegend.Legend._bold_title_patched = True

#1. Load pre-computed isoform fraction h5ad

In [3]:
adata = sc.read_h5ad(f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_ensemblids_annotatedonly_bc_anndata_preprocessed_with_isofrac_xgen_cdkn2a.h5ad")
print(f"Loaded: {adata.shape[0]} cells x {adata.shape[1]} isoforms")
print(f"Isoform fraction layer present: {'isoform_fraction' in adata.layers}")

# Add cell types and mapping
popv_to_ts = pd.read_csv("./../csvs/popv_to_ts_celltype_mapping.csv")
ts_celltypes = pd.read_csv("./../csvs/ts_celltypes.csv")
popv_to_ts_map = dict(zip(popv_to_ts["popv_prediction"], popv_to_ts["ts_cell_ontology_class"]))
ts_broad_map = dict(zip(ts_celltypes["cell_ontology_class"], ts_celltypes["broad_cell_class"]))


# Add tissue if not already present
if "tissue" not in adata.obs.columns:
    parts = adata.obs.index.str.split("_")
    adata.obs["tissue"] = ["_".join(p[1:]) if len(p) > 1 else "unknown" for p in parts]


# pigeon (structural category / associated gene) + SQANTI (coding/NMD) with prefixes
add_classification(adata, "./../csvs/all_samples_recollapsed_pigeon_classification_ensemblids.csv",
                   isoform_col="transcript_id", prefix="pigeon_", sep=",")
add_classification(adata, "./../csvs/all_samples_recollapsed_sqanti3_classification_ensemblids.csv",
                   isoform_col="transcript_id", prefix="sqanti_", sep=",")

print(f"Var columns: {list(adata.var.columns)}")
gc.collect()

2026-08-06 11:40:31,058 INFO Reading classification from ./../csvs/all_samples_recollapsed_pigeon_classification_ensemblids.csv


Loaded: 155117 cells x 130362 isoforms
Isoform fraction layer present: True


2026-08-06 11:40:54,528 INFO Classification table: 10200660 isoforms, columns ['chrom', 'strand', 'length', 'exons', 'structural_category', 'associated_gene', 'associated_transcript', 'ref_length', 'ref_exons', 'subcategory', 'FL', 'FSM_class', 'coding', 'predicted_NMD', 'isoform']
2026-08-06 11:40:54,528 WARNING Columns not found in classification file: {'filter_result'}
2026-08-06 11:41:04,054 INFO Matched 128739 / 130362 isoforms to classification
2026-08-06 11:41:10,141 INFO Added 14 columns; 128739 / 130362 isoforms annotated
2026-08-06 11:41:10,593 INFO Reading classification from ./../csvs/all_samples_recollapsed_sqanti3_classification_ensemblids.csv
2026-08-06 11:41:37,806 INFO Classification table: 10200660 isoforms, columns ['chrom', 'strand', 'length', 'exons', 'structural_category', 'subcategory', 'FSM_class', 'associated_gene', 'associated_transcript', 'ref_length', 'ref_exons', 'FL', 'coding', 'predicted_NMD', 'filter_result', 'isoform']
2026-08-06 11:41:47,785 INFO Match

Var columns: ['transcript_id', 'gene_name', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'pigeon_structural_category', 'pigeon_associated_gene', 'pigeon_associated_transcript', 'pigeon_subcategory', 'pigeon_chrom', 'pigeon_strand', 'pigeon_length', 'pigeon_exons', 'pigeon_coding', 'pigeon_FSM_class', 'pigeon_predicted_NMD', 'pigeon_FL', 'pigeon_ref_length', 'pigeon_ref_exons', 'sqanti_structural_category', 'sqanti_associated_gene', 'sqanti_associated_transcript', 'sqanti_subcategory', 'sqanti_chrom', 'sqanti_strand', 'sqanti_length', 'sqanti_exons', 'sqanti_coding', 'sqanti_FSM_class', 'sqanti_predicted_NMD', 'sqanti_FL', 'sqanti_ref_length', 'sqanti_ref_exons', 'sqanti_filter_result']


77

#2. CDKN2A classification

Classify cells as p16+, p14ARF+, Both, or CDKN2A- based on CDKN2A isoform expression.

In [4]:
# ── CDKN2A status from the xGen CDKN2A pulldown (raw counts) ─────────────────
# This object (built by 01_10) is subset to the 4 donors that have an xGen CDKN2A
# pulldown (TSP21/TSP25/TSP27/TSP33) and carries per-cell xGen columns. CDKN2A
# status is taken from the RAW pigeon-matrix pulldown counts
# (xgen_cdkn2a_raw_status), NOT the short-read call.
#   Senescent (CDKN2A+) = p16+ / Both only.
#   'p14ARF+'-only and 'other_CDKN2A+' cells are EXCLUDED (dropped).
#   'not_in_pulldown' (barcode not among the pulldown's called cells) -> CDKN2A-.
_status = adata.obs["xgen_cdkn2a_raw_status"].astype(str)
_drop = _status.isin(["other_CDKN2A+", "p14ARF+"])
n_excluded = int(_drop.sum())
adata = adata[(~_drop).values].copy()

_status = adata.obs["xgen_cdkn2a_raw_status"].astype(str).replace({"not_in_pulldown": "CDKN2A-"})
adata.obs["cdkn2a_status"] = pd.Categorical(
    _status.values, categories=["p16+", "Both", "CDKN2A-"]
)
adata.obs["is_senescent"] = adata.obs["cdkn2a_status"].isin(["p16+", "Both"]).values

sen_mask = adata.obs["is_senescent"].values
non_mask = ~sen_mask

print(f"Donors (xGen pulldown): {sorted(adata.obs['donor'].unique())}")
print(f"Excluded p14ARF+-only and other_CDKN2A+ cells: {n_excluded}")
print("\nCDKN2A status (from xGen raw pulldown counts):")
print(adata.obs["cdkn2a_status"].value_counts())
print(f"\nSenescent cells (p16+/Both): {int(sen_mask.sum())} / {adata.n_obs}")

Donors (xGen pulldown): ['TSP21', 'TSP25', 'TSP27', 'TSP33']
Excluded p14ARF+-only and other_CDKN2A+ cells: 935

CDKN2A status (from xGen raw pulldown counts):
cdkn2a_status
CDKN2A-    151048
Both         2565
p16+          569
Name: count, dtype: int64

Senescent cells (p16+/Both): 3134 / 154182


In [5]:
sen_by_ct = adata.obs[adata.obs["is_senescent"]].groupby("cell_ontology_class").size().sort_values(ascending=False)
print(f"\nSenescent cells per cell type (top 20):")
print(sen_by_ct.head(20))


Senescent cells per cell type (top 20):
cell_ontology_class
spermatocyte                                770
spermatid                                   563
plasma cell                                 391
macrophage                                  261
bladder urothelial cell                     166
fibroblast                                  116
b cell                                       96
cd8-positive, alpha-beta t cell              87
mesenchymal stem cell                        87
ciliated epithelial cell                     65
thymic fibroblast type 2                     64
stratified squamous epithelial cell          59
epithelial cell of uterus                    37
tracheal goblet cell                         28
regulatory t cell                            22
stromal cell                                 20
enterocyte of epithelium proper of ileum     20
cd4-positive, alpha-beta t cell              19
endothelial cell                             15
fast muscle cell           

#3. Gene column setup

In [6]:
gene_col = "pigeon_associated_gene" if "pigeon_associated_gene" in adata.var.columns else "gene_names"
print(f"Gene column for DIU: {gene_col}")
print(f"Isoform fraction layer: nnz={adata.layers['isoform_fraction'].nnz}")

# Build ENSG ID -> gene symbol mapping for readable labels
ensg_to_name = adata.var.dropna(subset=["pigeon_associated_gene", "gene_name"]).drop_duplicates("pigeon_associated_gene").set_index("pigeon_associated_gene")["gene_name"].to_dict()
print(f"ENSG -> gene name mapping: {len(ensg_to_name)} entries")

Gene column for DIU: pigeon_associated_gene
Isoform fraction layer: nnz=179534844
ENSG -> gene name mapping: 19189 entries


#4. Section A: DIU between senescent (CDKN2A+) vs non-senescent (CDKN2A-) per broad cell class

Run DIU within each broad cell class separately. This controls for cell-type composition
differences and identifies isoform switches that are truly senescence-associated rather than
driven by cell-type abundance shifts.

In [7]:
# ── Gene-level Dirichlet-multinomial DIU (fixes the one-sided per-isoform volcano) ──
# The per-isoform Wilcoxon volcano can look one-sided: a down-in-senescence isoform whose
# gene fails the two-sided min_cells/min_gene_frac floor is never tested -> absent from the
# result -> NaN after reindex -> matplotlib drops it. The DM test gives ONE p-value per gene
# on the isoform-COUNT composition (raw adata.X counts) and broadcasts it to each isoform,
# so gaining AND losing isoforms both appear. See differential_isoform_fraction.
from differential_isoform_fraction import (
    rank_gene_isoform_usage_dm_by_celltype, plot_gene_isoform_dm_volcano,
)
# Senescent vs Non-senescent, tested PER broad_cell_class (avoids the cell-type-mixture
# confound of a pooled test). Each cell type runs SERIALLY internally (fast per-gene, no
# inner pool overhead), while different cell types run in PARALLEL processes -- coarse-
# grained, so overhead is negligible vs the per-cell-type compute.
# NOTE: run this on a compute/JupyterLab allocation with DEDICATED cores and set N_JOBS to
# the core count. On a shared login node (no free cores) multiprocessing only adds overhead.
MIN_SEN_CELLS = 10   # broad cell classes needing at least this many senescent cells POOLED
MIN_PER_ARM_PER_DONOR = 5   # ...AND >=5 in EACH arm within at least ONE donor (see below)
N_JOBS = 16           # parallel cell types; set ~= dedicated cores available

adata.obs["sen_group"] = adata.obs["is_senescent"].map({True: "Senescent", False: "Non-senescent"})
# A pooled senescent count does not guarantee a within-donor contrast exists: a class can
# clear the pooled floor with its two arms coming from different donors, making the
# "senescence" effect a donor difference. Require one donor to supply BOTH arms. The cell
# after the CSV loader applies the identical rule post hoc -- identical because each split is
# fitted and BH-corrected independently, so dropping classes changes no retained p-value.
_bcc_sen = adata.obs[adata.obs["is_senescent"]].groupby("broad_cell_class").size()
_dc = (adata.obs.groupby(["broad_cell_class", "donor", "is_senescent"]).size()
       .unstack("is_senescent", fill_value=0))
for _c in (False, True):
    if _c not in _dc.columns:
        _dc[_c] = 0
_has_donor = ((_dc[False] >= MIN_PER_ARM_PER_DONOR)
              & (_dc[True] >= MIN_PER_ARM_PER_DONOR)).groupby("broad_cell_class").any()
dm_bccs = [b for b in _bcc_sen[_bcc_sen >= MIN_SEN_CELLS]
           .sort_values(ascending=False).index if _has_donor.get(b, False)]
print(f"Gene-level DM DIU for {len(dm_bccs)} broad cell classes (n_jobs={N_JOBS}): {dm_bccs}")

# One call: serial within each cell type, cell types in parallel. adata.X = raw counts.
dm_sen_all = rank_gene_isoform_usage_dm_by_celltype(
    adata, split_col="broad_cell_class", groupby="sen_group",
    group="Senescent", reference="Non-senescent",
    gene_col=gene_col, counts_layer=None,
    min_cells=10, include_untested=True, splits=dm_bccs, n_jobs=N_JOBS,
)

# Split back into per-cell-class frames for the volcano + hiding-diagnostic cells below.
dm_sen_frames = {b: g.reset_index(drop=True) for b, g in dm_sen_all.groupby("broad_cell_class")}
for bcc, dm in dm_sen_frames.items():
    _sig = dm.loc[dm["tested"], ["gene", "dm_pvals_adj"]].drop_duplicates()
    print(f"  {bcc}: {dm['gene'].nunique()} genes, "
          f"{(_sig['dm_pvals_adj'] < 0.05).sum()} significant (padj<0.05)")

# Save the per-cell-type (no-donor) DM results so the volcano/diagnostic cells below can be
# re-run from disk without recomputing the DM.
DM_CT_PATH = "./../csvs/06_04_dm_diu_per_celltype.csv"
dm_sen_all.to_csv(DM_CT_PATH, index=False)
print(f"Saved per-cell-type DM DIU -> {DM_CT_PATH} ({len(dm_sen_all)} rows)")

2026-08-06 11:49:25,272 INFO DM by broad_cell_class: 19 splits, 16 parallel workers


Gene-level DM DIU for 19 broad cell classes (n_jobs=16): ['male germ cell', 'lymphocyte of b lineage', 'myeloid leukocyte', 'fibroblast', 'transitional epithelial cell', 't cell', 'stem cell', 'ciliated epithelial cell', 'stratified epithelial cell', 'glandular epithelial cell', 'endo-epithelial cell', 'contractile cell', 'intestinal epithelial cell', 'stromal cell', 'endothelial cell', 'epithelial cell', 'innate lymphoid cell', 'epithelial cell of lung', 'vestibular dark cell']


2026-08-06 11:49:27,122 INFO DM isoform-usage: Senescent vs Non-senescent | 77 cells vs 351 | 18816 multi-isoform genes
2026-08-06 11:49:27,163 INFO DM isoform-usage: Senescent vs Non-senescent | 39 cells vs 672 | 18816 multi-isoform genes
2026-08-06 11:49:27,208 INFO DM isoform-usage: Senescent vs Non-senescent | 59 cells vs 1603 | 18816 multi-isoform genes
2026-08-06 11:49:27,324 INFO DM isoform-usage: Senescent vs Non-senescent | 32 cells vs 5122 | 18816 multi-isoform genes
2026-08-06 11:49:27,384 INFO DM isoform-usage: Senescent vs Non-senescent | 15 cells vs 3290 | 18816 multi-isoform genes
2026-08-06 11:49:27,455 INFO DM isoform-usage: Senescent vs Non-senescent | 28 cells vs 8271 | 18816 multi-isoform genes
2026-08-06 11:49:27,518 INFO DM isoform-usage: Senescent vs Non-senescent | 50 cells vs 4280 | 18816 multi-isoform genes
2026-08-06 11:49:27,562 INFO DM isoform-usage: Senescent vs Non-senescent | 1340 cells vs 4578 | 18816 multi-isoform genes
2026-08-06 11:49:27,565 INFO DM 

  ciliated epithelial cell: 11569 genes, 95 significant (padj<0.05)
  contractile cell: 14806 genes, 23 significant (padj<0.05)
  endo-epithelial cell: 12294 genes, 38 significant (padj<0.05)
  endothelial cell: 14470 genes, 0 significant (padj<0.05)
  epithelial cell: 13892 genes, 0 significant (padj<0.05)
  epithelial cell of lung: 12587 genes, 0 significant (padj<0.05)
  fibroblast: 15793 genes, 64 significant (padj<0.05)
  glandular epithelial cell: 14569 genes, 48 significant (padj<0.05)
  innate lymphoid cell: 10832 genes, 2 significant (padj<0.05)
  intestinal epithelial cell: 12806 genes, 103 significant (padj<0.05)
  lymphocyte of b lineage: 13794 genes, 1058 significant (padj<0.05)
  male germ cell: 14879 genes, 2723 significant (padj<0.05)
  myeloid leukocyte: 14984 genes, 769 significant (padj<0.05)
  stem cell: 15257 genes, 58 significant (padj<0.05)
  stratified epithelial cell: 11278 genes, 20 significant (padj<0.05)
  stromal cell: 14535 genes, 3 significant (padj<0.05)

In [8]:
# Read the saved per-cell-type (no-donor) DM results and rebuild the per-cell-class frames,
# so this cell runs standalone from disk (no need to re-run the DM compute cell above).
DM_CT_PATH = "./../csvs/06_04_dm_diu_per_celltype.csv"
dm_sen_all = pd.read_csv(DM_CT_PATH)
dm_sen_frames = {b: g.reset_index(drop=True) for b, g in dm_sen_all.groupby("broad_cell_class")}
print(f"Loaded per-cell-type DM DIU <- {DM_CT_PATH} "
      f"({len(dm_sen_all)} rows, {len(dm_sen_frames)} cell classes)")

Loaded per-cell-type DM DIU <- ./../csvs/18_dm_diu_per_celltype.csv (1553270 rows, 19 cell classes)
